In [4]:
import pandas as pd

df = pd.read_csv('data/train.csv')

In [5]:
# Examine the dataset structure
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst few rows:")
print(df.head())
print("\nTarget distribution:")
print(df['target'].value_counts())

Dataset shape: (7613, 5)

Column names: ['id', 'keyword', 'location', 'text', 'target']

First few rows:
   id keyword location                                               text  \
0   1     NaN      NaN  Our Deeds are the Reason of this #earthquake M...   
1   4     NaN      NaN             Forest fire near La Ronge Sask. Canada   
2   5     NaN      NaN  All residents asked to 'shelter in place' are ...   
3   6     NaN      NaN  13,000 people receive #wildfires evacuation or...   
4   7     NaN      NaN  Just got sent this photo from Ruby #Alaska as ...   

   target  
0       1  
1       1  
2       1  
3       1  
4       1  

Target distribution:
target
0    4342
1    3271
Name: count, dtype: int64


In [6]:
import keras
import keras_hub
import numpy as np
from sklearn.model_selection import train_test_split

# Extract features and labels
texts = df['text'].tolist()
labels = df['target'].tolist()

# Split the data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Training target distribution: {np.bincount(y_train)}")
print(f"Validation target distribution: {np.bincount(y_val)}")

# Create the pretrained classifier for binary classification
classifier = keras_hub.models.RobertaClassifier.from_preset(
    "roberta_base_en",
    num_classes=2,  # Binary classification: disaster (1) or not disaster (0)
)

# Compile the model
classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(5e-5),
    metrics=['accuracy'],
    jit_compile=True,
)

# Train the classifier
print("Training the model...")
history = classifier.fit(
    x=X_train,
    y=y_train,
    batch_size=16,
    epochs=3,
    validation_data=(X_val, y_val),
    verbose=1
)

Training set size: 6090
Validation set size: 1523
Training target distribution: [3473 2617]
Validation target distribution: [869 654]
Training the model...
Training the model...
Epoch 1/3
Epoch 1/3
381/381 ━━━━━━━━━━━━━━━━━━━━ 675s 2s/step - accuracy: 0.7916 - loss: 0.4685 - val_accuracy: 0.8313 - val_loss: 0.4118
Epoch 2/3
381/381 ━━━━━━━━━━━━━━━━━━━━ 675s 2s/step - accuracy: 0.7916 - loss: 0.4685 - val_accuracy: 0.8313 - val_loss: 0.4118
Epoch 2/3
381/381 ━━━━━━━━━━━━━━━━━━━━ 582s 2s/step - accuracy: 0.8493 - loss: 0.3679 - val_accuracy: 0.8050 - val_loss: 0.4962
Epoch 3/3
381/381 ━━━━━━━━━━━━━━━━━━━━ 582s 2s/step - accuracy: 0.8493 - loss: 0.3679 - val_accuracy: 0.8050 - val_loss: 0.4962
Epoch 3/3
381/381 ━━━━━━━━━━━━━━━━━━━━ 583s 2s/step - accuracy: 0.8888 - loss: 0.3029 - val_accuracy: 0.8175 - val_loss: 0.4725
381/381 ━━━━━━━━━━━━━━━━━━━━ 583s 2s/step - accuracy: 0.8888 - loss: 0.3029 - val_accuracy: 0.8175 - val_loss: 0.4725


In [8]:
# Evaluate the model on validation set
val_loss, val_accuracy = classifier.evaluate(X_val, y_val, verbose=0)
print(f"Final Validation Accuracy: {val_accuracy:.4f}")
print(f"Final Validation Loss: {val_loss:.4f}")

# Make predictions on some sample texts
sample_texts = [
    "Earthquake hits California, buildings collapsed",
    "I love sunny days at the beach",
    "Fire emergency evacuations in progress",
    "Had a great dinner with friends tonight"
]

predictions = classifier.predict(sample_texts)
predicted_classes = np.argmax(predictions, axis=1)

print("\nSample Predictions:")
for text, pred_class, confidence in zip(sample_texts, predicted_classes, predictions):
    disaster_prob = confidence[1]
    print(f"Text: '{text}'")
    print(f"Prediction: {'Disaster' if pred_class == 1 else 'Not Disaster'} (confidence: {disaster_prob:.3f})")
    print()

Final Validation Accuracy: 0.8175
Final Validation Loss: 0.4725
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step

Sample Predictions:
Text: 'Earthquake hits California, buildings collapsed'
Prediction: Disaster (confidence: 3.211)

Text: 'I love sunny days at the beach'
Prediction: Not Disaster (confidence: -1.095)

Text: 'Fire emergency evacuations in progress'
Prediction: Disaster (confidence: 3.189)

Text: 'Had a great dinner with friends tonight'
Prediction: Not Disaster (confidence: -1.551)



In [10]:


# Save the model weights (note: filename must end with .weights.h5)
model_save_path = 'roberta_disaster_classifier'
classifier.save_weights(f"{model_save_path}.weights.h5")
print(f"Model weights saved to: {model_save_path}.weights.h5")

Model weights saved to: roberta_disaster_classifier.weights.h5
